# 📓 EDA 탐색 노트북 — 상품군별 템플릿

`docs/EDA_GUIDE.md` **§3의 질문 11개**를 푸는 **작업장**입니다.

## 쓰는 법

1. 이 파일을 `notebooks/<domain>_<이름>.ipynb` 로 **복사**해서 쓰세요 (원본은 그대로 두기)
2. 아래 `DOMAIN` 을 담당 테이블로 바꾸고 위에서부터 실행
3. 질문별 셀에서 자유롭게 탐색 — 셀 추가/삭제 마음대로
4. **결론은 `docs/eda/<domain>_notes.md` 로 옮겨 적으세요** ← 이게 산출물입니다
5. 마지막 절에서 질의 20개 검증 + `eval/questions_<domain>.jsonl` 자동 생성

## ⚠️ 이 노트북은 산출물이 아닙니다

`.ipynb` 는 JSON + 출력셀 + 실행카운터라 **git diff 가 안 읽혀 리뷰가 불가능**하고,
탐색 과정이 전부 남아 **결론이 안 보입니다**.

> **노트북 = 작업장 / `_notes.md` + `.jsonl` = 산출물**

개인 노트북은 `.gitignore` 되어 있습니다 (이 템플릿만 커밋됨).

## 🔒 원본 DB 훼손 방지

다음 셀에서 **읽기 전용 커넥션**을 만들고, 쓰기가 실제로 막혔는지 확인합니다.
노트북은 셀을 순서 없이 재실행하는 게 자연스러워서 스크립트보다 위험합니다 —
`df.to_sql(...)` 오타 하나로 테이블이 날아갈 수 있습니다. 이 셀을 **반드시 먼저** 실행하세요.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 안전 셋업 — 반드시 먼저 실행
# ═══════════════════════════════════════════════════════════
import sqlite3, json, collections
from pathlib import Path
import pandas as pd

# ← 담당 테이블로 변경
DOMAIN = "domestic_etfs"
#   domestic_bonds / domestic_etfs / overseas_etfs / public_funds

ROOT = Path.cwd()
if not (ROOT / "data").exists():          # notebooks/ 안에서 열었을 때
    ROOT = ROOT.parent
DB = (ROOT / "data" / "financial_products.db").resolve()
assert DB.exists(), f"DB 없음: {DB}\n먼저 `python scripts/build_db.py` 실행"

# 🔒 읽기 전용 커넥션 — 쓰기 시도는 커넥션 레벨에서 거부됩니다
conn = sqlite3.connect(f"{DB.as_uri()}?mode=ro", uri=True)

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 200)

# 쓰기가 실제로 막혔는지 확인 (안 막혔으면 여기서 멈춤)
try:
    conn.execute("CREATE TABLE __probe__(x)")
    raise RuntimeError("❌ 쓰기가 허용됩니다 — 커넥션 설정을 확인하세요!")
except sqlite3.OperationalError as e:
    assert "readonly" in str(e), e
    print(f"🔒 읽기 전용 확인 — 원본 훼손 불가 ({e})")

TOTAL = conn.execute(f"SELECT COUNT(*) FROM {DOMAIN}").fetchone()[0]
COLS = [r[1] for r in conn.execute(f'PRAGMA table_info("{DOMAIN}")')]
print(f"📊 {DOMAIN}: {TOTAL:,}행 × {len(COLS)}컬럼")

## 헬퍼

`TRIM()` 이 기본으로 적용됩니다 — §8-① 패딩 함정 때문에 **맨손 정확일치는 조용히 실패**합니다.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 자주 쓰는 헬퍼
# ═══════════════════════════════════════════════════════════

def q(sql, params=()):
    """SQL 실행 → DataFrame"""
    return pd.read_sql_query(sql, conn, params=params)


def vals(col, table=None, limit=50):
    """distinct 값 + 건수 (TRIM 적용). 범주형 컬럼 파악용."""
    t = table or DOMAIN
    return q(f"""SELECT TRIM(CAST("{col}" AS TEXT)) AS value, COUNT(*) AS n
                 FROM {t}
                 WHERE TRIM(COALESCE(CAST("{col}" AS TEXT), '')) <> ''
                 GROUP BY 1 ORDER BY n DESC LIMIT {limit}""")


def miss(col, table=None):
    """결측 3종 내역 — NULL / 공백문자열 / 패딩 (§8-②)"""
    t = table or DOMAIN
    return q(f"""SELECT COUNT(*) AS total,
                   SUM(CASE WHEN "{col}" IS NULL THEN 1 ELSE 0 END) AS n_null,
                   SUM(CASE WHEN "{col}" IS NOT NULL
                             AND TRIM(CAST("{col}" AS TEXT))='' THEN 1 ELSE 0 END) AS n_blank,
                   SUM(CASE WHEN CAST("{col}" AS TEXT)
                             <> TRIM(CAST("{col}" AS TEXT)) THEN 1 ELSE 0 END) AS n_padded
                 FROM {t}""")


def peek(col, n=10, table=None):
    """실제 저장값 샘플. 패딩이 눈에 보이도록 repr 로 출력."""
    t = table or DOMAIN
    rows = q(f'SELECT "{col}" AS v FROM {t} WHERE "{col}" IS NOT NULL LIMIT {n}')
    for v in rows["v"]:
        print(repr(v))


def cross(col, other_table, other_col):
    """
    §3 질문 5 — 우리 테이블의 값이 다른 테이블과 같은 표기를 쓰는지.
    겹치는 값 / 우리만 있는 값 / 상대만 있는 값을 보여줍니다.
    """
    a = set(vals(col, limit=100000)["value"])
    b = set(vals(other_col, table=other_table, limit=100000)["value"])
    print(f"{DOMAIN}.{col}: {len(a)}종   {other_table}.{other_col}: {len(b)}종")
    print(f"  ✅ 겹침      {len(a & b):>5}종  {sorted(a & b)[:8]}")
    print(f"  ◀ 우리만    {len(a - b):>5}종  {sorted(a - b)[:8]}")
    print(f"  ▶ 상대만    {len(b - a):>5}종  {sorted(b - a)[:8]}")
    return a, b


def kor(col):
    """한글 컬럼명"""
    r = q("SELECT korean_name FROM schema_metadata WHERE table_name=? AND column_name=?",
          (DOMAIN, col))
    return r["korean_name"][0] if len(r) else ""


print("헬퍼 준비 완료: q() vals() miss() peek() cross() kor()")

## 1단계 산출물 확인 — 프로파일러가 이미 찾아둔 것

**같은 걸 다시 찾지 마세요.** 여기 없는 것을 찾는 게 이 노트북의 목적입니다.
(안 떠 있으면 먼저 `python scripts/profile_table.py <DOMAIN>` 실행)

In [ ]:
# ═══════════════════════════════════════════════════════════
# 프로파일러 발견 요약
# ═══════════════════════════════════════════════════════════
import yaml

auto_path = ROOT / "ontology" / "enums" / f"{DOMAIN}.auto.yaml"
assert auto_path.exists(), f"{auto_path.name} 없음 — `python scripts/profile_table.py {DOMAIN}` 먼저 실행"
AUTO = yaml.safe_load(auto_path.read_text(encoding="utf-8"))

for f in AUTO.get("table_findings", []):
    print(f"🔴 [테이블] {f['detector']}: {f['message']}")

# 컬럼 그룹 — "전부 해당"은 그룹의 성질, "일부만"이 진짜 이상치
for g in AUTO.get("group_findings", []):
    print(f"\n▪ 그룹 {g['group']} ({g['size']}개)")
    for t in g.get("traits", []):
        print(f"    ─ {t['detector']}: 전부 해당 → 그룹 성질 (조치 불필요)")
    for e in g.get("exceptions", []):
        print(f"    ⚠️ {e['detector']}: {e['ratio']:.0%} — {', '.join(e['columns'][:5])}")
    if g.get("value_shapes"):
        print(f"    ⚠️ 값 표현 {len(g['value_shapes'])}가지로 갈림")

# 실제 값이 절반 미만인 컬럼 — 답변 정책이 필요한 컬럼
thin = [(c, e["values_present"], e["total"]) for c, e in AUTO["columns"].items()
        if e["total"] and e["values_present"] / e["total"] < 0.5]
print("\n📉 실제 값 < 50% 컬럼")
display(pd.DataFrame(sorted(thin, key=lambda r: r[1]),
                     columns=["column", "values_present", "total"]).head(15))

# 결측 여부 판정이 필요한 값 — <domain>.yaml 의 missing_semantics 에 선언
pend = [(c, j["value"], j["count"], j["hint"])
        for c, e in AUTO["columns"].items() for j in e.get("judgment_needed", [])]
if pend:
    print("\n❓ 결측 판정 필요 (EDA_GUIDE §3 🕳️)")
    display(pd.DataFrame(sorted(pend, key=lambda r: -r[2]),
                         columns=["column", "value", "count", "hint"]).head(10))

---

# §3 — 답해야 할 질문 11개

각 절의 결론을 **`docs/eda/<domain>_notes.md`** 에 옮겨 적으세요.
셀은 자유롭게 추가하세요. 아래 코드는 출발점일 뿐입니다.

---

## 🔹 개체 (Entity)

### Q1. 내 상품군에서 "상품 하나"를 식별하는 것은 무엇인가?

> 자명해 보여도 확인하세요. 공모펀드는 `std_itm_no` 가 PK가 아니었습니다 —
> 같은 펀드가 속성코드별로 최대 16행입니다.

**메모:** _(여기에 답을 적으세요)_

In [ ]:
# 식별자 후보 컬럼들의 유일성 확인
for c in COLS:
    if any(k in c.lower() for k in ("no", "cd", "id", "isin")):
        n = conn.execute(f'SELECT COUNT(DISTINCT "{c}") FROM {DOMAIN}').fetchone()[0]
        if n > 100:
            mark = "  ✅ 유일" if n == TOTAL else f"  ⚠️ 중복 (평균 {TOTAL/n:.1f}배)"
            print(f"{c:<28} distinct {n:>7,} / {TOTAL:,}{mark}")

### Q2. 내 테이블에 "상품이 아닌 개체"가 섞여 있는가?

> 국내ETF마스터에는 ETN이 30.7% 들어 있습니다. 같은 개체로 볼 것인가, 나눌 것인가?

**메모:**

In [ ]:
# 상품 유형을 가르는 컬럼이 있는지 (저카디널리티 범주형 훑기)
for c in COLS:
    n = conn.execute(f'SELECT COUNT(DISTINCT TRIM(CAST("{c}" AS TEXT))) FROM {DOMAIN}').fetchone()[0]
    if 2 <= n <= 8:
        print(f"── {c}  ({kor(c)})")
        print(vals(c, limit=8).to_string(index=False), "\n")

---

## 🔹 관계 (Relation)

### Q3. 내 상품군의 상품은 무엇과 연결되는가?

> 후보: 운용사 · 발행사 · 기초지수 · 벤치마크 · 상장시장 · 기초자산 · 통화 · 구성종목 …
> **어떤 컬럼으로** 그 연결이 들어 있는지 함께 적어주세요.

**메모:**

| 연결 대상 | 우리 테이블의 컬럼 | 비고 |
| :--- | :--- | :--- |
|  |  |  |

In [ ]:
# 한글 컬럼명으로 "연결" 성격의 컬럼 훑기
meta = q("SELECT column_name, korean_name FROM schema_metadata WHERE table_name=?", (DOMAIN,))
display(meta[meta["korean_name"].str.contains(
    "사|회사|운용|발행|지수|벤치마크|시장|통화|국가|지역|자산", na=False)])

### Q4. 그 연결 대상 중 다른 상품군에도 등장할 것은? ★

> **워크샵의 핵심 재료입니다. 여기서 공통 축이 *발견*됩니다.**
> 예: 채권 발행사와 ETF 운용사가 같은 금융그룹일 수 있습니다.

**메모:**

### Q5. 우리 테이블의 그 값이 다른 테이블과 같은 표기를 쓰고 있을까?

> 이미 확인된 불일치는 `EDA_GUIDE.md` §5-A 에 있습니다 — **거기 없는 것**을 찾아주세요.

**메모:**

In [ ]:
# cross() 로 두 테이블의 같은 축 값이 겹치는지 확인
# 예시 — 담당 테이블에 맞게 바꿔 쓰세요
# cross("cu_fund_mgmt_co", "overseas_etfs", "cu_fund_mgmt_co")
# cross("wu_inv_rgn",      "public_funds",  "fd_ivst_rgn_desc")
# cross("wu_inv_ast_type", "overseas_etfs", "wu_inv_ast_type")

---

## 🔹 분류 (Classification)

### Q6. 내 상품군을 분류하는 방식이 몇 가지이고, 무엇이 1차 분류인가?

> 자산군? 운용전략? 투자지역? 위험등급? 여러 축이 있다면 서로 직교하는지 겹치는지.

**메모:**

### Q7. 그 분류에 계층이 있는가?

> 예: 일본 ⊂ 아시아 ⊂ 글로벌. 계층이 있으면 온톨로지의 `rdfs:subClassOf` 가 실제로 값어치를 합니다.

**메모:**

In [ ]:
# 두 분류 축이 직교하는지 교차표로 확인
# 예시 — 담당 테이블에 맞게
# display(q(f"""SELECT TRIM(wu_inv_ast_type) AS 자산군, TRIM(wu_inv_rgn) AS 지역, COUNT(*) AS n
#               FROM {DOMAIN} GROUP BY 1,2 ORDER BY n DESC LIMIT 30""")
#           .pivot(index="자산군", columns="지역", values="n").fillna(0).astype(int))

---

## 🔹 고유 개념 & 한계

### Q8. 내 상품군에만 있고 다른 상품군엔 없는 개념은?

> 채권: 만기·듀레이션·신용등급 / ETF: 괴리율·추적오차·합성 vs 실물 / 펀드: 클래스 체계·재간접
> → 온톨로지에서 공통 상위 클래스가 아니라 **하위 클래스 고유 속성**이 됩니다.

**메모:**

### Q9. 사용자가 물을 법한데 지금 데이터로 답할 수 없는 것은?

> 이게 그대로 `확인할 수 없음` 응답 정책이 됩니다.

**메모:**

### Q10. 비전공자 팀원이 이 상품군 질의를 이해하려면 최소한 알아야 할 도메인 지식은?

> 워크샵에서 다른 트랙 담당자에게 설명한다고 생각하고 적어주세요.

**메모:**

In [ ]:
# Q9 재료 — 답변 불가 컬럼 목록 (실제 값이 10% 미만)
rows = [(c, e["values_present"], e["total"], e["korean_name"])
        for c, e in AUTO["columns"].items() if e["values_present"] < e["total"] * 0.1]
display(pd.DataFrame(sorted(rows, key=lambda r: r[1]),
                     columns=["column", "values_present", "total", "korean_name"]))

---

## 🔹 함정

### Q11. §8에 없는 새로운 데이터 함정을 발견했다면?

> **재현 SQL + 영향 건수**를 함께 적어주세요.
>
> 💡 그게 **다른 테이블에도 기계적으로 돌릴 수 있는 패턴**이면,
> `scripts/profile_table.py` 에 탐지기로 추가해 PR 보내주세요 (`EDA_GUIDE.md` §2).
> 한 사람의 발견이 4개 테이블 전체에 자동 적용됩니다.

**메모:**

In [ ]:
# 자유 탐색

---

# §4 — 예상 질의 20개

아래 `QUESTIONS` 리스트를 채우면 **검증 → jsonl 생성**이 자동으로 됩니다.
손으로 JSON 을 쓰지 마세요 — 문법 오류와 0건 반환이 여기서 바로 드러납니다.

**배분 목표**

| 항목 | 목표 |
| :--- | :--- |
| 총 문항 | 20 |
| 난이도 | 하 7 / 중 7 / 상 6 |
| `unanswerable` + `clarify` | **6개 이상** |
| `qtype` | 조건검색3 / 정보조회3 / 비교3 / 연산·순위3 / 교차상품군2 / 네거티브3 / 결측2 / 역질문1 |

> `교차상품군` 2문항은 **지금은 답이 안 나오는 게 정상**입니다.
> 그 질의가 워크샵에서 "이 관계를 온톨로지에 넣어야 한다"의 근거가 됩니다.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 질의 20개 — 아래 예시를 지우고 채우세요
# ═══════════════════════════════════════════════════════════
QUESTIONS = [
    dict(
        qid="ETF-D-001", difficulty="중", qtype="연산·순위", expected_behavior="answer",
        question="국내 상장 ETF 중 최근 1년 수익률이 가장 높은 채권형 ETF 3개를 알려줘.",
        gold_sql="""SELECT TRIM(pd_nm) AS pd_nm, du_er_1y
                    FROM domestic_etfs
                    WHERE pd_grp_no='ETF' AND TRIM(wu_inv_ast_type)='채권'
                      AND du_er_1y IS NOT NULL
                    ORDER BY du_er_1y DESC LIMIT 3""",
        must_include=["ETF명 3개", "1년 수익률 수치"],
        must_not_include=["수익률 전망", "매수 추천"],
        source_columns=["pd_grp_no", "wu_inv_ast_type", "du_er_1y"],
        note="pd_grp_no 필터 없으면 ETN이 섞임. du_er_1y 결측 20.6% / 크로스체크: ",
    ),
    dict(
        qid="ETF-D-018", difficulty="상", qtype="네거티브(미존재)", expected_behavior="unanswerable",
        question="KODEX AI 로봇 ETF의 총보수가 얼마야?",
        gold_sql=None,
        must_include=["확인할 수 없음", "해당 종목 없음"],
        must_not_include=["0.4%", "총보수는"],
        source_columns=[],
        note="2026-07-11 기준 미존재 상품. 멘토 경고 사항 직결. / 크로스체크: ",
    ),
    dict(
        qid="ETF-D-020", difficulty="중", qtype="역질문필요", expected_behavior="clarify",
        question="안전한 ETF 하나 추천해줘.",
        gold_sql=None,
        must_include=["위험등급", "투자 지역", "확인이 필요"],
        must_not_include=["추천드립니다", "가장 안전한"],
        source_columns=[],
        note="조건 부족 → 역질문 분기. 과제 명세의 '조건부 안내' 요구사항. / 크로스체크: ",
    ),
]
print(f"{len(QUESTIONS)}문항 작성됨 (목표 20)")

## 검증 — DoD: *"gold_sql 을 직접 실행해 결과를 눈으로 확인"*

아래 셀이 **전 문항의 `gold_sql` 을 실제로 실행**하고 결과를 보여줍니다.
❌ 가 하나도 없어야 통과입니다.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 질의 검증
# ═══════════════════════════════════════════════════════════
problems = []
for item in QUESTIONS:
    qid, beh, sql = item["qid"], item["expected_behavior"], item.get("gold_sql")

    if beh == "answer":
        if not sql:
            problems.append(f"{qid}: expected_behavior=answer 인데 gold_sql 이 없음"); continue
        try:
            df = q(sql)
        except Exception as e:
            problems.append(f"{qid}: SQL 오류 — {e}"); continue
        if len(df) == 0:
            problems.append(f"{qid}: 0건 반환 — 조건을 확인하세요"); continue
        print(f"✅ {qid}  {len(df)}건 — {item['question'][:45]}")
        display(df.head(5))
    else:
        if sql:
            problems.append(f"{qid}: {beh} 인데 gold_sql 이 채워져 있음 (None 이어야 함)")
        else:
            print(f"✅ {qid}  ({beh}) — {item['question'][:45]}")

print("\n" + "═" * 60)
diff = collections.Counter(i["difficulty"] for i in QUESTIONS)
beh = collections.Counter(i["expected_behavior"] for i in QUESTIONS)
qt = collections.Counter(i["qtype"] for i in QUESTIONS)
neg = beh["unanswerable"] + beh["clarify"]

# DoD 충족 여부 — 이것도 통과해야 산출물을 낼 수 있습니다
shortfalls = []
if len(QUESTIONS) != 20:
    shortfalls.append(f"총 문항 {len(QUESTIONS)} / 20")
if (diff["하"], diff["중"], diff["상"]) != (7, 7, 6):
    shortfalls.append(f"난이도 배분 하{diff['하']}/중{diff['중']}/상{diff['상']} (목표 7/7/6)")
if neg < 6:
    shortfalls.append(f"unanswerable+clarify {neg} / 6 이상")
if any("크로스체크" in i.get("note", "") and i.get("note", "").rstrip().endswith(":")
       for i in QUESTIONS):
    shortfalls.append("크로스체크 승인자 이름이 비어 있는 문항 있음")

mark = lambda ok: "✅" if ok else "❌"
print(f"총 문항   {len(QUESTIONS):>2} / 20        {mark(len(QUESTIONS) == 20)}")
print(f"난이도    하{diff['하']} 중{diff['중']} 상{diff['상']}   (목표 하7/중7/상6) "
      f"{mark((diff['하'], diff['중'], diff['상']) == (7, 7, 6))}")
print(f"answer 외 {neg:>2} / 6 이상     {mark(neg >= 6)}")
print(f"qtype     {dict(qt)}")

if problems:
    print("\n❌ SQL 문제 — 반드시 고쳐야 합니다:")
    for p in problems:
        print("   -", p)
if shortfalls:
    print("\n⚠️  DoD 미충족:")
    for s in shortfalls:
        print("   -", s)
if not problems and not shortfalls:
    print("\n🎉 전 항목 통과 — 다음 셀에서 jsonl 을 생성하세요.")

## 산출물 생성

검증을 통과했으면 실행하세요. `eval/questions_<domain>.jsonl` 이 만들어집니다.

- **SQL 문제**가 있으면 무조건 막힙니다 (고쳐야 함)
- **DoD 미충족**(문항 수·난이도 배분 등)이면 막히되, 중간 저장이 필요하면 `ALLOW_PARTIAL = True`

> ⚠️ **2인 크로스체크가 남아 있습니다.** 다른 담당자 1명이 `gold_sql` 을 재실행해 승인하고,
> 각 문항의 `note` 끝에 승인자 이름을 남겨야 2단계 완료입니다.

In [ ]:
# ═══════════════════════════════════════════════════════════
# eval/questions_<domain>.jsonl 생성
# ═══════════════════════════════════════════════════════════
ALLOW_PARTIAL = False      # 작업 중 중간 저장이 필요하면 True

if problems:
    raise RuntimeError(f"❌ SQL 문제 {len(problems)}건 — 위 검증 셀을 먼저 해결하세요")
if shortfalls and not ALLOW_PARTIAL:
    raise RuntimeError(
        f"❌ DoD 미충족 {len(shortfalls)}건: {shortfalls}\n"
        f"   완성 후 다시 실행하거나, 중간 저장이면 ALLOW_PARTIAL = True 로 두세요"
    )
if shortfalls:
    print(f"⚠️  DoD 미충족 상태로 저장합니다 (ALLOW_PARTIAL=True): {shortfalls}\n")

out = ROOT / "eval" / f"questions_{DOMAIN}.jsonl"
out.parent.mkdir(parents=True, exist_ok=True)
with out.open("w", encoding="utf-8") as fp:
    for item in QUESTIONS:
        row = dict(item)
        if row.get("gold_sql"):
            row["gold_sql"] = " ".join(row["gold_sql"].split())   # 한 줄로 정리
        fp.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"✅ {out.relative_to(ROOT)} — {len(QUESTIONS)}문항 저장")
print("\n다음 할 일:")
print("  1. docs/eda/<domain>_notes.md 에 §3 질문 11개의 답 정리")
print("  2. 질의 20문항 2인 크로스체크 (note 에 승인자 이름)")
print("  3. §3 질문 5·9·11 의 답을 T0(리드)에 공유 — 워크샵 안건용")
print("  4. PR: 브랜치 eda/<domain>  (노트북은 커밋되지 않습니다)")